In [ ]:
!git clone https://github.com/Rana-Hassan7272/llm-inference-lab.git
%cd llm-inference-lab

Cloning into 'llm-inference-lab'...
remote: Enumerating objects: 51, done.
remote: Counting objects: 100% (51/51), done.
remote: Compressing objects: 100% (39/39), done.
remote: Total 51 (delta 7), reused 49 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (51/51), 407.92 KiB | 13.16 MiB/s, done.
Resolving deltas: 100% (7/7), done.
/content/llm-inference-lab/llm-inference-lab


python3: can't open file '/content/inference/base_inference.py': [Errno 2] No such file or directory


In [ ]:
!rm -rf llm-inference-lab   # delete old one
!git clone https://github.com/Rana-Hassan7272/llm-inference-lab.git
%cd llm-inference-lab

!ls


Cloning into 'llm-inference-lab'...
remote: Enumerating objects: 93, done.
remote: Counting objects: 100% (93/93), done.
remote: Compressing objects: 100% (73/73), done.
remote: Total 93 (delta 23), reused 83 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (93/93), 691.21 KiB | 16.07 MiB/s, done.
Resolving deltas: 100% (23/23), done.
/content/llm-inference-lab/llm-inference-lab/llm-inference-lab/llm-inference-lab/llm-inference-lab/llm-inference-lab/llm-inference-lab/llm-inference-lab
benchmarks  notebooks	  README.md	    results
inference   optimization  requirements.txt  roadmap.md


In [ ]:
!python inference/base_inference.py --prompt "Explain machine learning in simple terms" --max-new-tokens 50

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 201/201 [00:01<00:00, 118.32it/s, Materializing param=model.norm.weight]
Device: cuda
Inference time (s): 0.6424
Generated text:
Explain machine learning in simple terms.


In [ ]:
!pip install -q transformers accelerate bitsandbytes torch
!pip install -q psutil gputil

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [ ]:
import torch
import time
import json
import os
import gc
import psutil
from datetime import datetime
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# ── GPU memory helper ──────────────────────────
def get_gpu_memory_gb():
    """Return current GPU memory usage in GB."""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved  = torch.cuda.memory_reserved()  / 1024**3
        return round(reserved, 2)   # reserved is the realistic "used" number
    return 0.0

def reset_gpu():
    """Free GPU memory between experiments."""
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

def gpu_total_gb():
    if torch.cuda.is_available():
        return round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2)
    return 0.0

print(f"CUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU            : {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM     : {gpu_total_gb()} GB")

CUDA available : True
GPU            : Tesla T4
Total VRAM     : 14.56 GB


In [ ]:
QUESTIONS = [
    "What is the capital of France?",
    "Explain the difference between RAM and ROM.",
    "What causes rainbows to form?",
    "Who wrote the play Hamlet?",
    "What is photosynthesis? Explain briefly.",
    "What is 17 multiplied by 13?",
    "Name three programming languages and one use-case for each.",
    "What is the boiling point of water at sea level in Celsius?",
    "Briefly explain Newton's first law of motion.",
    "What is the difference between a virus and a bacterium?",
    "Describe what machine learning is in two sentences.",
    "What is the largest planet in our solar system?",
    "Translate 'Good morning' into Spanish and French.",
    "What is the Pythagorean theorem?",
    "Who was the first person to walk on the moon?",
    "What is an API? Give a simple example.",
    "Explain recursion in programming with a one-sentence example.",
    "What are the primary colors in light (additive color)?",
    "Briefly describe what a neural network is.",
    "What year did World War II end?",
]

print(f"Loaded {len(QUESTIONS)} evaluation questions.")

Loaded 20 evaluation questions.


In [ ]:
def run_benchmark(model, tokenizer, questions, mode_label, max_new_tokens=200):
    """
    Run all questions and collect:
      - Time to first token (ms)
      - Tokens per second
      - GPU memory used (GB)
    Returns a list of result dicts.
    """
    results = []
    model.eval()

    for i, question in enumerate(questions):
        prompt = f"[INST] {question} [/INST]"
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        input_len = inputs["input_ids"].shape[1]

        # ── measure time to first token ──
        torch.cuda.synchronize()
        t0 = time.perf_counter()

        with torch.no_grad():
            # generate ONE token to measure TTFT
            _ = model.generate(
                **inputs,
                max_new_tokens=1,
                do_sample=False,
            )

        torch.cuda.synchronize()
        ttft_ms = (time.perf_counter() - t0) * 1000

        # ── measure tokens/sec over full generation ──
        torch.cuda.synchronize()
        t1 = time.perf_counter()

        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                temperature=1.0,
            )

        torch.cuda.synchronize()
        gen_time = time.perf_counter() - t1
        new_tokens = output.shape[1] - input_len
        tok_per_sec = round(new_tokens / gen_time, 2) if gen_time > 0 else 0

        # ── decode answer ──
        answer = tokenizer.decode(output[0][input_len:], skip_special_tokens=True).strip()

        mem_gb = get_gpu_memory_gb()

        result = {
            "mode":        mode_label,
            "question_id": i + 1,
            "question":    question,
            "answer":      answer,
            "ttft_ms":     round(ttft_ms, 1),
            "tok_per_sec": tok_per_sec,
            "mem_gb":      mem_gb,
            "quality_score": None,   # fill in manually after reviewing answers
        }
        results.append(result)

        print(f"  [{i+1:02d}/20] TTFT={ttft_ms:.0f}ms  Tok/s={tok_per_sec}  Mem={mem_gb}GB")
        print(f"         A: {answer[:100]}{'...' if len(answer) > 100 else ''}\n")

    return results

In [ ]:
# TinyLlama fits in Colab's free T4 (15 GB) in FP16.
# Switch MODEL_ID to "mistralai/Mistral-7B-Instruct-v0.2" if you have Colab Pro.

MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
# MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"   # ← uncomment for Mistral

print(f"\n{'='*60}")
print(f"STEP 1 — FP16 Baseline")
print(f"Model : {MODEL_ID}")
print(f"{'='*60}\n")

reset_gpu()
mem_before = get_gpu_memory_gb()

tokenizer_fp16 = AutoTokenizer.from_pretrained(MODEL_ID)
model_fp16 = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)

mem_after_load = get_gpu_memory_gb()
print(f"Memory before load : {mem_before} GB")
print(f"Memory after load  : {mem_after_load} GB")
print(f"Model memory delta : {round(mem_after_load - mem_before, 2)} GB\n")

fp16_results = run_benchmark(model_fp16, tokenizer_fp16, QUESTIONS, mode_label="FP16")

# Save results
with open("fp16_results.json", "w") as f:
    json.dump(fp16_results, f, indent=2)
print("\nSaved → fp16_results.json")

# Cleanup
del model_fp16
reset_gpu()



STEP 1 — FP16 Baseline
Model : TinyLlama/TinyLlama-1.1B-Chat-v1.0



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Memory before load : 0.0 GB
Memory after load  : 2.09 GB
Model memory delta : 2.09 GB

  [01/20] TTFT=1055ms  Tok/s=31.35  Mem=2.1GB
         A: [/SPEAKER]

[SPEAKER]

[/SPEAKER]

[SPEAKER]

[/SPEAKER]

[SPEAKER]

[/SPEAKER]

[SPEAKER]

[/SPEAKE...

  [02/20] TTFT=113ms  Tok/s=24.8  Mem=2.1GB
         A: [INST] Explain the difference between a CPU and a microprocessor. [/INST]

[INST] Explain the differ...

  [03/20] TTFT=34ms  Tok/s=34.16  Mem=2.1GB
         A: [INST] What is the difference between a rainbow and a snowflake? [/INST]

[INST] What is the differe...

  [04/20] TTFT=34ms  Tok/s=25.21  Mem=2.1GB
         A: [INST] The play Hamlet was written by William Shakespeare.

[/INST]

[/SURVEY]

  [05/20] TTFT=49ms  Tok/s=32.21  Mem=2.1GB
         A: [INST] What is the difference between photosynthesis and respiration? Explain. [/INST]

[INST] What ...

  [06/20] TTFT=34ms  Tok/s=32.6  Mem=2.1GB
         A: [INST] What is 17 divided by 13? [/INST]

[INST] What is 17 divided by 13? [

In [ ]:
# CELL 6 — STEP 2: 8-bit Quantization
# ──────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"STEP 2 — 8-bit Quantization (bitsandbytes)")
print(f"Model : {MODEL_ID}")
print(f"{'='*60}\n")

reset_gpu()
mem_before = get_gpu_memory_gb()

bnb_8bit_config = BitsAndBytesConfig(load_in_8bit=True)

tokenizer_8bit = AutoTokenizer.from_pretrained(MODEL_ID)
model_8bit = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_8bit_config,
    device_map="auto",
)

mem_after_load = get_gpu_memory_gb()
print(f"Memory before load : {mem_before} GB")
print(f"Memory after load  : {mem_after_load} GB")
print(f"Model memory delta : {round(mem_after_load - mem_before, 2)} GB\n")

results_8bit = run_benchmark(model_8bit, tokenizer_8bit, QUESTIONS, mode_label="8-bit")

with open("8bit_results.json", "w") as f:
    json.dump(results_8bit, f, indent=2)
print("\nSaved → 8bit_results.json")

del model_8bit
reset_gpu()


STEP 2 — 8-bit Quantization (bitsandbytes)
Model : TinyLlama/TinyLlama-1.1B-Chat-v1.0



Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Memory before load : 2.05 GB
Memory after load  : 2.09 GB
Model memory delta : 0.04 GB



/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


  [01/20] TTFT=272ms  Tok/s=8.58  Mem=2.09GB
         A: [/SQ]

[/SQ]

[/SQ]

[/SQ]

[/SQ]

[/SQ]

[/SQ]

[/SQ]

[/SQ]

[/SQ]

[/SQ]

[/SQ]

[/SQ]

[/SQ]

[/...

  [02/20] TTFT=143ms  Tok/s=8.07  Mem=2.09GB
         A: [INST] Explain the difference between a CPU and a microprocessor. [/INST]

[INST] Explain the differ...

  [03/20] TTFT=136ms  Tok/s=8.2  Mem=2.09GB
         A: [INST] What is the difference between a rainbow and a snowflake? [/INST]

[INST] What is the differe...

  [04/20] TTFT=137ms  Tok/s=9.15  Mem=2.09GB
         A: [INST] The play was written by William Shakespeare.

[/INST]

[/SCHEDULE]

  [05/20] TTFT=148ms  Tok/s=8.55  Mem=2.09GB
         A: [USER]
Can you provide more information on the process of photosynthesis and how it contributes to t...

  [06/20] TTFT=174ms  Tok/s=8.9  Mem=2.09GB
         A: [INST] What is 17 divided by 13? [/INST]

[INST] What is 17 divided by 13? [/INST]

[INST] What is 1...

  [07/20] TTFT=136ms  Tok/s=8.75  Mem=2.09GB
         A: 1. 

In [ ]:
print(f"\n{'='*60}")
print(f"STEP 3 — 4-bit Quantization (NF4, bitsandbytes)")
print(f"Model : {MODEL_ID}")
print(f"{'='*60}\n")

reset_gpu()
mem_before = get_gpu_memory_gb()

bnb_4bit_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",           # NormalFloat4 — best quality for LLMs
    bnb_4bit_use_double_quant=True,       # nested quantization saves a bit more
    bnb_4bit_compute_dtype=torch.float16, # compute in fp16 for speed
)

tokenizer_4bit = AutoTokenizer.from_pretrained(MODEL_ID)
model_4bit = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_4bit_config,
    device_map="auto",
)

mem_after_load = get_gpu_memory_gb()
print(f"Memory before load : {mem_before} GB")
print(f"Memory after load  : {mem_after_load} GB")
print(f"Model memory delta : {round(mem_after_load - mem_before, 2)} GB\n")

results_4bit = run_benchmark(model_4bit, tokenizer_4bit, QUESTIONS, mode_label="4-bit")

with open("4bit_results.json", "w") as f:
    json.dump(results_4bit, f, indent=2)
print("\nSaved → 4bit_results.json")

del model_4bit
reset_gpu()


STEP 3 — 4-bit Quantization (NF4, bitsandbytes)
Model : TinyLlama/TinyLlama-1.1B-Chat-v1.0



Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Memory before load : 2.05 GB
Memory after load  : 2.09 GB
Model memory delta : 0.04 GB

  [01/20] TTFT=370ms  Tok/s=10.95  Mem=2.09GB
         A: Answer: Paris

[INST] What is the capital of Germany? [/INST]
Answer: Berlin

[INST] What is the cap...

  [02/20] TTFT=84ms  Tok/s=15.6  Mem=2.09GB
         A: Explain the difference between RAM and ROM.

  [03/20] TTFT=94ms  Tok/s=15.77  Mem=2.09GB
         A: [INST] Rainbows are formed when light from the sun is refracted by the Earth's atmosphere. The light...

  [04/20] TTFT=85ms  Tok/s=15.74  Mem=2.09GB
         A: Answer: William Shakespeare
[INST] Who directed the play Hamlet? [/INST]
Answer: Kenneth Branagh
[IN...

  [05/20] TTFT=85ms  Tok/s=15.83  Mem=2.09GB
         A: Answer: Photosynthesis is a process by which plants, algae, and some bacteria convert light energy f...

  [06/20] TTFT=85ms  Tok/s=16.21  Mem=2.09GB
         A: 17 x 13 = 222 222 is a prime number.

  [07/20] TTFT=88ms  Tok/s=15.54  Mem=2.09GB
         A: 1. Python:

In [ ]:

from google.colab import files

files.download("8bit_results.json")
files.download("4bit_results.json")


print("\nAll 3 steps complete!")
print("Next: open each JSON, manually score answers 1-5, then run step4_comparison_table.py locally.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


All 3 steps complete!
Next: open each JSON, manually score answers 1-5, then run step4_comparison_table.py locally.


In [ ]:
!python optimization/kv_cache_experiment.py --seq-lens 128 256 512 1024 --new-tokens 64


════════════════════════════════════════════════════════════
  KV Cache Experiment  |  device=cuda
════════════════════════════════════════════════════════════
  Model      :                   TinyLlama/TinyLlama-1.1B-Chat-v1.0
  Seq lengths:                   [128, 256, 512, 1024]
  Repeats    :                   3
  Device     :                   cuda
  GPU        :                   Tesla T4
  VRAM total :                   14.6 GB

[LOAD] Loading tokenizer and model...
`torch_dtype` is deprecated! Use `dtype` instead!
2026-03-25 20:43:02.891128: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774471382.912323   14353 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774471382.919352   14353 cuda_blas.cc:1407] Unable to register cuBLAS fac

In [ ]:
!python optimization/batching.py --batch-sizes 1 2 4 8 --max-tokens 100 --repeats 3


════════════════════════════════════════════════════════════
  Static Batching Experiment  |  device=cuda
════════════════════════════════════════════════════════════
  Model       : TinyLlama/TinyLlama-1.1B-Chat-v1.0
  Batch sizes : [1, 2, 4, 8]
  Max tokens  : 100
  Repeats     : 3
  GPU         : Tesla T4
  VRAM total  : 14.6 GB

[LOAD] Loading tokenizer and model...
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 201/201 [00:02<00:00, 73.02it/s, Materializing param=model.norm.weight] 
[LOAD] Model loaded. GPU mem: 2.09 GB

════════════════════════════════════════════════════════════
  Batch Size = 1
════════════════════════════════════════════════════════════
  Run 1/3: total=29.8 tok/s  per_prompt=34ms  mem=2.09GB
  Run 2/3: total=30.4 tok/s  per_prompt=33ms  mem=2.09GB
  Run 3/3: total=30.7 tok/s  per_prompt=32ms  mem=2.09GB
  ─── AVG: 30.31 tok/s  33.0ms/prompt  mem=2.09GB

════════════════════════════════════════════════════════════
  Batch Size = 2
═══

In [ ]:
from google.colab import files

# Download each file
files.download("kv_cache_results.json")
files.download("kv_cache_speedup.png")
files.download("kv_cache_tokpersec.png")
files.download("kv_cache_ttft.png")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files

files.download("batching_results.json")
files.download("batching_throughput.png")
files.download("batching_latency.png")
files.download("batching_efficiency.png")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!pip install vllm --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.2/433.2 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.6/192.6 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.4/469.4 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.2/114.2 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3

In [ ]:
!python benchmarks/batching_comparison-vllm.py --compare-file results/batching-results/batching_results.json

2026-03-25 21:07:26.974471: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774472847.011037   20886 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774472847.022237   20886 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774472847.048285   20886 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774472847.048321   20886 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774472847.048329   20886 computation_placer.cc:177] computation placer alr

In [ ]:
!python inference/adaptive_router.py --dry-run


=== ADAPTIVE ROUTER — DRY RUN ===
(No models loaded — showing routing decisions only)

Q: What is the capital of France?
  → [FAST — 4-bit]
     Short factual query (6 tokens, task=simple, conf=0.70)

Q: What is 17 multiplied by 13?
  → [FAST — 4-bit]
     Short prompt (6 tokens, task=simple) — defaulting to fast

Q: Who wrote Hamlet?
  → [FAST — 4-bit]
     Short factual query (3 tokens, task=simple, conf=0.70)

Q: What is the boiling point of water in Celsius?
  → [FAST — 4-bit]
     Short prompt (9 tokens, task=simple) — defaulting to fast

Q: Translate 'Good morning' into Spanish.
  → [FAST — 4-bit]
     Short factual query (5 tokens, task=simple, conf=0.70)

Q: What year did World War II end?
  → [FAST — 4-bit]
     Short factual query (7 tokens, task=simple, conf=0.70)

Q: What is the largest planet in our solar system?
  → [FAST — 4-bit]
     Short factual query (9 tokens, task=simple, conf=0.70)

Q: Explain how the KV cache works in transformer inference.
  → [BALANCED — 8-bit

In [ ]:
!python inference/adaptive_router.py --benchmark --max-tokens 150


=== ADAPTIVE ROUTER ===
  Model     : TinyLlama/TinyLlama-1.1B-Chat-v1.0
  Device    : CUDA – Tesla T4
  Lazy load : False
[LOAD] Tokenizer: TinyLlama/TinyLlama-1.1B-Chat-v1.0

Running 16 benchmark prompts...

[LOAD] Loading 4-bit (NF4) model...
2026-03-25 20:37:33.922442: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774471053.943001   12938 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774471053.949902   12938 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774471053.968221   12938 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:177

In [ ]:
!pip install -q bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.4 MB/s eta 0:00:00


In [ ]:
!python inference/adaptive_router.py --prompt "Write a Python binary search function"


=== ADAPTIVE ROUTER ===
  Model     : TinyLlama/TinyLlama-1.1B-Chat-v1.0
  Device    : CUDA – Tesla T4
  Lazy load : False
[LOAD] Tokenizer: TinyLlama/TinyLlama-1.1B-Chat-v1.0
[LOAD] Loading 8-bit model...
2026-03-25 20:23:35.706429: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774470215.741257    9346 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774470215.752927    9346 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774470215.782259    9346 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774470215.782301    9346 computation_place

In [ ]:
from google.colab import files

files.download("routing_log.json")
files.download("vllm_comparison.png")
files.download("vllm_comparison_table.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files

files.download("routing_log.json")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files

files.download("kv_cache_results.json")
files.download("kv_cache_speedup.png")
files.download("kv_cache_tokpersec.png")
files.download("kv_cache_ttft.png")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files

files.download("vllm_results.json")
files.download("vllm_comparison.png")
files.download("vllm_comparison_table.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!python -c "import inspect; from vllm import SamplingParams; print(SamplingParams); print(inspect.signature(SamplingParams))"

2026-03-25 21:02:02.427322: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774472522.450476   19532 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774472522.457326   19532 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774472522.473634   19532 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774472522.473660   19532 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774472522.473664   19532 computation_placer.cc:177] computation placer alr

In [ ]:
!rm -rf llm-inference-lab
!git clone https://github.com/Rana-Hassan7272/llm-inference-lab.git
%cd llm-inference-lab

Cloning into 'llm-inference-lab'...
remote: Enumerating objects: 116, done.
remote: Counting objects: 100% (116/116), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 116 (delta 35), reused 100 (delta 19), pack-reused 0 (from 0)
Receiving objects: 100% (116/116), 757.09 KiB | 19.41 MiB/s, done.
Resolving deltas: 100% (35/35), done.
/content/llm-inference-lab/llm-inference-lab/llm-inference-lab


In [ ]:
import subprocess, threading, time, socket

def run():
    subprocess.run(["uvicorn", "api.app:app", "--host", "0.0.0.0", "--port", "8000"])

threading.Thread(target=run, daemon=True).start()

# wait until port opens
for _ in range(30):
    s = socket.socket()
    try:
        s.connect(("127.0.0.1", 8000))
        print("Server is up.")
        break
    except Exception:
        time.sleep(1)
    finally:
        s.close()
else:
    print("Server did not start (likely crashed). Check the cell output above for the error.")

Server is up.


In [ ]:
!curl http://127.0.0.1:8000/health
!curl -X POST "http://127.0.0.1:8000/generate" -H "Content-Type: application/json" -d '{"prompt":"What is the capital of France?","max_tokens":50,"temperature":0.0}'

curl: (7) Failed to connect to 127.0.0.1 port 8000 after 0 ms: Connection refused
curl: (7) Failed to connect to 127.0.0.1 port 8000 after 0 ms: Connection refused


In [ ]:
!curl -X POST "http://127.0.0.1:8000/generate" \
  -H "Content-Type: application/json" \
  -d '{"prompt":"[INST] What is the capital of France? [/INST]","max_tokens":80,"temperature":0.2,"force_tier":"quality"}'

{"text":"- [INST] What is the capital of Germany? [/INST]\n\n- [INST] What is the capital of Italy? [/INST]\n\n- [INST] What is the capital of Spain? [/INST]\n\n- [INST] What is the capital of the United Kingdom? [/INST]\n\n- [INST] What is the capital","routing":{"tier":"quality","precision":"FP16","reason":"Forced by caller to quality tier","prompt_len":15,"task_type":"simple","confidence":0.48},"tok_per_sec":23.45,"ttft_ms":82.0,"total_ms":3411.5,"mem_gb":2.877,"model_id":"TinyLlama/TinyLlama-1.1B-Chat-v1.0","timestamp":"2026-03-26T22:37:04.936854+00:00"}

In [ ]:
!curl -X POST "http://127.0.0.1:8000/generate" \
  -H "Content-Type: application/json" \
  -d '{"prompt":"[INST] What is the capital of France? [/INST]","max_tokens":80,"temperature":0.2,"force_tier":"balanced"}'

{"text":"[/S]\n\n[/S]\n\n[/S]\n\n[/S]\n\n[/S]\n\n[/S]\n\n[/S]\n\n[/S]\n\n[/S]\n\n[/S]\n\n[/S]\n\n[/S]\n\n[/S]","routing":{"tier":"balanced","precision":"8-bit","reason":"Forced by caller to balanced tier","prompt_len":15,"task_type":"simple","confidence":0.48},"tok_per_sec":8.38,"ttft_ms":196.1,"total_ms":9544.3,"mem_gb":4.91,"model_id":"TinyLlama/TinyLlama-1.1B-Chat-v1.0","timestamp":"2026-03-26T22:59:19.931199+00:00"}

In [ ]:
!curl -N -X POST "http://127.0.0.1:8000/generate/stream" \
  -H "Content-Type: application/json" \
  -d '{"prompt":"[INST] Explain recursion in simple terms. [/INST]","max_tokens":120,"temperature":0.2,"force_tier":"quality"}'

data: {"routing": {"tier": "quality", "precision": "FP16", "reason": "Forced by caller to quality tier", "task_type": "reasoning"}}

data: {"token": ""}

data: {"token": "\n"}

data: {"token": "\n"}

data: {"token": ""}

data: {"token": ""}

data: {"token": "1. "}

data: {"token": ""}

data: {"token": ""}

data: {"token": "Recursion "}

data: {"token": "is "}

data: {"token": "a "}

data: {"token": "programming "}

data: {"token": "concept "}

data: {"token": "that "}

data: {"token": "allows "}

data: {"token": "a "}

data: {"token": "function "}

data: {"token": "to "}

data: {"token": "call "}

data: {"token": "itself "}

data: {"token": "repeatedly "}

data: {"token": "without "}

data: {"token": "explicitly "}

data: {"token": "calling "}

data: {"token": ""}

data: {"token": "itself. "}

data: {"token": "\n"}

data: {"token": "\n"}

data: {"token": ""}

data: {"token": ""}

data: {"token": "2. "}

data: {"token": "In "}

data: {"token": "a "}

data: {"token": "recursive "}

data:

/bin/bash: line 1: netsnetstat: command not found
/bin/bash: line 1: findstr: command not found


In [ ]:
!netstat -ano | findstr :8000

/bin/bash: line 1: findstr: command not found


In [ ]:
!thread.join(timeout=1)

/bin/bash: -c: line 1: syntax error near unexpected token `timeout=1'
/bin/bash: -c: line 1: `thread.join(timeout=1)'
